# Week 7 — Explainability: "Why Did the Model Say That?"

**Learning Goals:**
- Build explanations that are simple, consistent, and actionable
- Compare explanation methods: SHAP, saliency, integrated gradients
- Standardize output for the Copilot agent

**Methods:**
- SHAP (for sklearn baselines)
- Gradient saliency (for deep models)
- Integrated gradients (more robust)
- Temporal saliency (which time steps matter?)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '../src')
from data_loader import load_train, load_test
from preprocess import preprocess_pipeline, INFORMATIVE_SENSORS_FD001
from models.cnn_transformer import CNNTransformerModel
from explain import (
    compute_saliency,
    compute_integrated_gradients,
    compute_temporal_saliency,
    format_explanation,
)

plt.style.use('seaborn-v0_8-whitegrid')
device = 'cpu'  # Gradients work best on CPU for small batches
print('Imports OK')

In [ ]:
# Load data and model
df_train = load_train(fd_number=1, rul_cap=125)
df_test, rul_true = load_test(fd_number=1)
data = preprocess_pipeline(df_train, df_test, window_size=30)

# Load trained model (from Week 6 checkpoint)
import os
ckpt_path = '../checkpoints/cnn_transformer_FD001_best.pt'
n_features = data['config']['n_features']

model = CNNTransformerModel(n_features=n_features, seq_len=30)
if os.path.exists(ckpt_path):
    model.load_state_dict(torch.load(ckpt_path, map_location='cpu'))
    print('Loaded checkpoint from Week 6')
else:
    print('No checkpoint found — using random weights for demo')

sensors = INFORMATIVE_SENSORS_FD001

## 1. Gradient Saliency

In [ ]:
# Explain 3 engines
sample_engines = [0, 49, 99]

for eng_idx in sample_engines:
    x = data['X_test'][eng_idx]
    expl = compute_saliency(model, x, sensors, device='cpu', top_k=5)
    
    print(f'\n--- Engine {eng_idx+1} (True RUL={rul_true[eng_idx]}) ---')
    print(f'Method: {expl["method"]}')
    for s in expl['top_sensors']:
        print(f'  #{s["rank"]}: {s["sensor"]:12s} importance={s["importance"]:.4f}')

## 2. Integrated Gradients

In [ ]:
# Compare saliency vs integrated gradients
eng_idx = 0
x = data['X_test'][eng_idx]

sal_expl = compute_saliency(model, x, sensors, device='cpu')
ig_expl = compute_integrated_gradients(model, x, sensors, device='cpu')

# Side by side comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, expl, title in zip(axes, [sal_expl, ig_expl], ['Gradient Saliency', 'Integrated Gradients']):
    attribs = expl['all_attributions']
    names = list(attribs.keys())
    values = list(attribs.values())
    sorted_idx = np.argsort(values)[::-1]
    ax.barh([names[i] for i in sorted_idx], [values[i] for i in sorted_idx], color='steelblue')
    ax.set_title(f'{title} — Engine {eng_idx+1}')
    ax.set_xlabel('Attribution Score')
    ax.invert_yaxis()

plt.tight_layout()
plt.show()

## 3. Temporal Saliency

In [ ]:
# Which time steps are most important?
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, eng_idx in zip(axes, sample_engines):
    x = data['X_test'][eng_idx]
    temporal_imp = compute_temporal_saliency(model, x, device='cpu')
    
    ax.bar(range(len(temporal_imp)), temporal_imp, color='coral', edgecolor='black', alpha=0.7)
    ax.set_xlabel('Time Step in Window')
    ax.set_ylabel('Importance')
    ax.set_title(f'Engine {eng_idx+1} — Temporal Saliency')

plt.suptitle('Which time steps matter most?', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Explanation Consistency Analysis

In [ ]:
# Check if the same sensors appear as top contributors across engines
from collections import Counter

top_sensor_counts = Counter()
for i in range(len(data['X_test'])):
    expl = compute_saliency(model, data['X_test'][i], sensors, device='cpu', top_k=3)
    for s in expl['top_sensors']:
        top_sensor_counts[s['sensor']] += 1

print('Top sensors across all test engines (top-3 per engine):')
for sensor, count in top_sensor_counts.most_common(10):
    print(f'  {sensor:12s}: appears in top-3 for {count}/{len(data["X_test"])} engines')

# Plot
plt.figure(figsize=(10, 5))
mc = top_sensor_counts.most_common()
plt.bar([x[0] for x in mc], [x[1] for x in mc], color='steelblue', edgecolor='black')
plt.xticks(rotation=45, ha='right')
plt.ylabel('Count (in top-3)')
plt.title('Most Important Sensors Across All Test Engines')
plt.tight_layout()
plt.show()